In [34]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Генерация данных с шумом

X, y = make_classification(
    n_samples=1200, 
    n_features=50,
    n_informative=10,
    n_redundant=10,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [35]:
# 2. Определение архитектуры MLP

class MLP(nn.Module):
    def __init__(self, input_dim: int, use_dropout: bool = False, dropout_p: float = 0.5):
        super().__init__()
        
        layers = [
            nn.Linear(input_dim, 128),
            nn.ReLU(),
        ]
        
        if use_dropout:
            layers.append(nn.Dropout(p=dropout_p))
            
        layers.extend([
            nn.Linear(128, 64),
            nn.ReLU(),
        ])
        
        if use_dropout:
            layers.append(nn.Dropout(p=dropout_p))
            
        layers.append(nn.Linear(64, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x).squeeze(-1)

In [36]:
# 3. Цикл обучения 
def train_and_evaluate(model, train_loader, val_loader, optimizer, criterion, epochs=80):
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * len(batch_y)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            train_correct += (preds == batch_y).sum().item()

        model.eval()
        val_loss, val_correct = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                
                val_loss += loss.item() * len(batch_y)
                preds = (torch.sigmoid(logits) >= 0.5).float()
                val_correct += (preds == batch_y).sum().item()
                
        if epoch % 20 == 0 or epoch == epochs:
            print(f"Epoch {epoch}")
            print(f"Train Loss: {train_loss / len(train_loader.dataset):.4f}, Acc: {train_correct / len(train_loader.dataset):.4f}")
            print(f"Val Loss: {val_loss / len(val_loader.dataset):.4f}, Acc: {val_correct / len(val_loader.dataset):.4f}")

In [37]:
# 4. Эксперименты
criterion = nn.BCEWithLogitsLoss()

print("Модель без регуляризации")
model_overfit = MLP(input_dim=50, use_dropout=False)
opt_overfit = torch.optim.Adam(model_overfit.parameters(), lr=1e-3, weight_decay=0.0)
train_and_evaluate(model_overfit, train_loader, val_loader, opt_overfit, criterion)

print("\nМодель с Dropout и Weight Decay")
model_reg = MLP(input_dim=50, use_dropout=True, dropout_p=0.4)
opt_reg = torch.optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-2)
train_and_evaluate(model_reg, train_loader, val_loader, opt_reg, criterion)

Модель без регуляризации
Epoch 20
Train Loss: 0.0051, Acc: 1.0000
Val Loss: 0.4177, Acc: 0.8694
Epoch 40
Train Loss: 0.0007, Acc: 1.0000
Val Loss: 0.5340, Acc: 0.8722
Epoch 60
Train Loss: 0.0002, Acc: 1.0000
Val Loss: 0.5948, Acc: 0.8694
Epoch 80
Train Loss: 0.0001, Acc: 1.0000
Val Loss: 0.6405, Acc: 0.8694

Модель с Dropout и Weight Decay
Epoch 20
Train Loss: 0.1240, Acc: 0.9571
Val Loss: 0.2609, Acc: 0.8750
Epoch 40
Train Loss: 0.0754, Acc: 0.9845
Val Loss: 0.2587, Acc: 0.8722
Epoch 60
Train Loss: 0.0623, Acc: 0.9881
Val Loss: 0.2620, Acc: 0.8944
Epoch 80
Train Loss: 0.0539, Acc: 0.9952
Val Loss: 0.2609, Acc: 0.8833
